# PINN Sampling Strategy Comparison: RAR vs Curriculum-Enhanced

This trains two FourierPINN models on the same geometry, same architecture, same seed,
differing only in `sampling_strategy` for the RAR (residual-adaptive refinement)
collocation updates. That isolates the sampling strategy as the single variable.

Strategies compared:
- `rar`: original residual-proportional RAR-D (Lu et al. 2021 style). Always samples new
  collocation points proportional to \|PDE residual\| as soon as physics loss activates.
- `curriculum`: Curriculum-Enhanced Adaptive Sampling (MDPI, Dec 2025 style). Stays at pure
  uniform/stratified sampling for a warmup fraction of training, then blends toward
  residual-weighted sampling. The argument is that early-training residuals are dominated
  by random-init noise, not real physics difficulty, so trusting them too early wastes
  exploration budget.

See `src/pinn/sampling.py` for the full literature citations and implementation.

## Kaggle Dataset Setup

Same two datasets as the other PINN notebooks:

1. Source code: upload `src/` as a Kaggle dataset (slug suggestion: `thermo-pinn-src`)
2. 3D-ICE training data: upload `.npz` files from `data/3d-ice/` (slug suggestion: `3dice-thermal-data`)

Expected time: about 2x a single PINN run (two full trainings back-to-back), e.g. 4-8h on
a T4 at the default 8000 epochs for geometry1. Reduce `EPOCHS` in the config cell for a
faster demonstration run.


In [ ]:
import subprocess, sys

# Install any missing packages (PyYAML is the only non-standard dep)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)

import os
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import sys
from pathlib import Path

# ── Kaggle dataset paths ────────────────────────────────────────────────────
# Adjust these slugs to match the datasets you attached to this notebook.
SRC_DATASET   = 'thermo-pinn-src'    # dataset containing the src/ folder
DATA_DATASET  = '3dice-thermal-data' # dataset containing the .npz files

SRC_ROOT  = Path(f'/kaggle/input/{SRC_DATASET}')
DATA_ROOT = Path(f'/kaggle/input/{DATA_DATASET}')
OUT_DIR   = Path('/kaggle/working/checkpoints/pinn_sampling_comparison')

# Add source root to path so we can import src.*
sys.path.insert(0, str(SRC_ROOT))

# Verify
assert SRC_ROOT.exists(),  f'Source dataset not found at {SRC_ROOT}. Check SRC_DATASET slug.'
assert DATA_ROOT.exists(), f'Data dataset not found at {DATA_ROOT}. Check DATA_DATASET slug.'
print('Source root:', SRC_ROOT)
print('Data root:  ', DATA_ROOT)

In [ ]:
import logging
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.core.geometry_builders import get_geometry_by_name
from src.pinn.data_loader import ThermalDataset, NormStats, compute_norm_stats
from src.pinn.model import build_model
from src.pinn.trainer import Trainer

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
print('Imports OK')

In [ ]:
GEOM_NAME      = 'geometry1'
FOURIER_SIGMA  = 10.0    # Fourier feature bandwidth (10 for geometry1, no TSVs)
HIDDEN_DIM     = 256
N_RES_BLOCKS   = 6
N_COL          = 20000
EPOCHS         = 8000    # reduce (e.g. 2000) for a quicker demonstration run
LR             = 3e-4
LOG_INTERVAL   = 200
VAL_INTERVAL   = 1

STRATEGY_A = 'rar'
STRATEGY_B = 'curriculum'
SEED       = 42          # shared seed; isolates sampling_strategy as the only variable

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')
print(f'Comparing sampling strategies: {STRATEGY_A!r} vs {STRATEGY_B!r}')
print(f'Config: hidden={HIDDEN_DIM} blocks={N_RES_BLOCKS} n_col={N_COL} epochs={EPOCHS}')

In [ ]:
geometry = get_geometry_by_name(GEOM_NAME)
geometries = {GEOM_NAME: geometry}
print(geometry.summary())

In [ ]:
def collect_files(data_dir: Path, geom: str, split: str):
    files = sorted(data_dir.rglob(f'{geom}_{split}_*.npz'))
    if not files:
        files = sorted(data_dir.glob(f'{geom}_{split}_*.npz'))
    return files

In [ ]:
train_files = collect_files(DATA_ROOT, GEOM_NAME, 'train')
test_files  = collect_files(DATA_ROOT, GEOM_NAME, 'test')

assert train_files, f'No training files found in {DATA_ROOT}. Check DATA_DATASET slug and file naming.'
print(f'Training files : {len(train_files)}')
print(f'Test files     : {len(test_files)}')
print('First train file:', train_files[0].name)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
norm_path = OUT_DIR / 'norm_stats.json'

if norm_path.exists():
    norm_stats = NormStats.load(norm_path)
    print('Loaded existing norm stats from', norm_path)
else:
    print('Computing norm stats over', len(train_files), 'training files...')
    norm_stats = compute_norm_stats(train_files, geometries)
    norm_stats.save(norm_path)
    print('Saved norm stats to', norm_path)

print(f'T range    : [{norm_stats.T_min:.1f}, {norm_stats.T_max:.1f}] K')
print(f'geom extents: {norm_stats.geom_extents}')

In [ ]:
print('Loading training dataset...')
train_dataset = ThermalDataset(train_files, geometries, norm_stats)

val_files = test_files if test_files else train_files[-3:]
print('Loading validation dataset...')
val_dataset = ThermalDataset(val_files, geometries, norm_stats)

print(f'Train: {len(train_dataset)} scenarios')
print(f'Val  : {len(val_dataset)} scenarios')

In [ ]:
def train_with_strategy(strategy: str, seed: int = SEED):
    """Build a fresh model and Trainer with the given sampling_strategy, train, return (trainer, best_ckpt)."""
    torch.manual_seed(seed)

    out_dir = OUT_DIR / f'strategy_{strategy}'
    out_dir.mkdir(parents=True, exist_ok=True)

    n_layers = len(geometry.layers)
    model = build_model(
        n_layers=n_layers,
        fourier_sigma=FOURIER_SIGMA,
        hidden_dim=HIDDEN_DIM,
        n_res_blocks=N_RES_BLOCKS,
        device=DEVICE,
    )

    trainer = Trainer(
        model=model,
        geometry=geometry,
        norm_stats=norm_stats,
        train_data=train_dataset,
        val_data=val_dataset,
        output_dir=out_dir,
        n_col=N_COL,
        epochs=EPOCHS,
        lr=LR,
        device=DEVICE,
        log_interval=LOG_INTERVAL,
        val_interval=VAL_INTERVAL,
        sampling_strategy=strategy,
    )
    print(f'\n=== Training with sampling_strategy={strategy!r} ===')
    best_ckpt = trainer.train()
    print(f'Done. Best val MAE: {min(trainer.history["val_mae_K"]):.3f} K')
    return trainer, best_ckpt

In [ ]:
trainer_a, ckpt_a = train_with_strategy(STRATEGY_A)

In [ ]:
trainer_b, ckpt_b = train_with_strategy(STRATEGY_B)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Sampling Strategy Comparison: {GEOM_NAME}, {STRATEGY_A} vs {STRATEGY_B}', fontsize=13)

ax = axes[0]
ax.semilogy(trainer_a.history['epoch'], trainer_a.history['train_total'], label=f'{STRATEGY_A} (total loss)', color='steelblue')
ax.semilogy(trainer_b.history['epoch'], trainer_b.history['train_total'], label=f'{STRATEGY_B} (total loss)', color='tomato')
ax.set_xlabel('Epoch'); ax.set_ylabel('Total training loss (log scale)')
ax.set_title('Training loss'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(trainer_a.history['epoch'], trainer_a.history['val_mae_K'], label=STRATEGY_A, color='steelblue')
ax.plot(trainer_b.history['epoch'], trainer_b.history['val_mae_K'], label=STRATEGY_B, color='tomato')
best_a = min(trainer_a.history['val_mae_K']); best_b = min(trainer_b.history['val_mae_K'])
ax.axhline(best_a, linestyle='--', color='steelblue', alpha=0.4)
ax.axhline(best_b, linestyle='--', color='tomato', alpha=0.4)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val MAE (K)')
ax.set_title(f'Validation MAE  ({STRATEGY_A}: {best_a:.2f}K, {STRATEGY_B}: {best_b:.2f}K)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'sampling_strategy_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def plot_collocation_scatter(trainer, strategy_name, ax):
    coords = trainer._col_coords.detach().cpu().numpy()
    order = np.arange(len(coords))  # chronological position (oldest -> newest)
    sca = ax.scatter(coords[:, 0], coords[:, 1], c=order, cmap='viridis', s=3, alpha=0.6)
    ax.set_title(f'{strategy_name}: collocation points (n={len(coords)})\ncolor = order added (dark=initial, bright=late RAR)')
    ax.set_xlabel('x (norm)'); ax.set_ylabel('y (norm)'); ax.set_aspect('equal')
    return sca

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
sca_a = plot_collocation_scatter(trainer_a, STRATEGY_A, axes[0])
sca_b = plot_collocation_scatter(trainer_b, STRATEGY_B, axes[1])
plt.colorbar(sca_a, ax=axes[0], label='chronological order')
plt.colorbar(sca_b, ax=axes[1], label='chronological order')
plt.suptitle('Collocation point placement: where did each strategy add points?', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'collocation_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'{STRATEGY_A}: {len(trainer_a._col_coords)} final collocation points (started at {N_COL})')
print(f'{STRATEGY_B}: {len(trainer_b._col_coords)} final collocation points (started at {N_COL})')

In [ ]:
def evaluate(ckpt_path, model, label):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    T_range = norm_stats.T_max - norm_stats.T_min
    results = []
    with torch.no_grad():
        for sc in val_dataset.scenarios:
            htc_t  = torch.tensor(sc.htc_norm,   dtype=torch.float32, device=DEVICE)
            tamb_t = torch.tensor(sc.t_amb_norm, dtype=torch.float32, device=DEVICE)
            tsv_t  = torch.tensor(sc.tsv_frac,   dtype=torch.float32, device=DEVICE)
            T_pred_norm = model(sc.coords, sc.layer_ids, sc.power, htc_t, tamb_t, tsv_t)
            T_pred_K = T_pred_norm.cpu().numpy() * T_range + norm_stats.T_min
            T_true_K = sc.temp.cpu().numpy()     * T_range + norm_stats.T_min
            err = np.abs(T_pred_K - T_true_K)
            results.append({'name': sc.name, 'mae_K': float(err.mean()),
                             'p95_K': float(np.percentile(err, 95)), 'max_K': float(err.max())})
    maes = [r['mae_K'] for r in results]
    print(f'\n[{label}]  mean MAE={np.mean(maes):.3f}K  max MAE={np.max(maes):.3f}K')
    return results

results_a = evaluate(ckpt_a, trainer_a.model, STRATEGY_A)
results_b = evaluate(ckpt_b, trainer_b.model, STRATEGY_B)

print(f'\n{"Scenario":<30} {STRATEGY_A+" MAE":>14} {STRATEGY_B+" MAE":>14} {"Delta":>10}')
print('-' * 72)
for ra, rb in zip(results_a, results_b):
    delta = ra['mae_K'] - rb['mae_K']
    print(f'{ra["name"]:<30} {ra["mae_K"]:>14.3f} {rb["mae_K"]:>14.3f} {delta:>10.3f}')

In [ ]:
import json

summary = {
    'geometry': GEOM_NAME,
    'strategy_a': STRATEGY_A,
    'strategy_b': STRATEGY_B,
    'epochs': EPOCHS,
    'seed': SEED,
    'best_val_mae_K': {STRATEGY_A: min(trainer_a.history['val_mae_K']), STRATEGY_B: min(trainer_b.history['val_mae_K'])},
    'test_mae_mean_K': {STRATEGY_A: float(np.mean([r['mae_K'] for r in results_a])),
                         STRATEGY_B: float(np.mean([r['mae_K'] for r in results_b]))},
    'final_collocation_count': {STRATEGY_A: len(trainer_a._col_coords), STRATEGY_B: len(trainer_b._col_coords)},
    'checkpoints': {STRATEGY_A: str(ckpt_a), STRATEGY_B: str(ckpt_b)},
}

with open(OUT_DIR / 'sampling_comparison_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))